In [ ]:
import pandas as pd
import tensorflow as tf
import os
from sklearn.model_selection import train_test_split
from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras.layers import Input, Dense, GlobalAveragePooling3D, Concatenate, Dropout
from tensorflow.keras.models import Model
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping

# ==========================================
# 1. จัดการ Path และโหลดข้อมูล
# ==========================================
SURVEY_CSV = 'data/data_from_questionaire.csv'
IG_CSV = 'data/data_from_intragram.csv'

SURVEY_IMG_DIR = 'data/questionnaire_images/'
IG_IMG_DIR = 'data/instagram_images/'

df_survey = pd.read_csv(SURVEY_CSV)
df_ig = pd.read_csv(IG_CSV)

# สร้าง Full Path ให้รูปภาพแบบสอบถาม (อยู่ในโฟลเดอร์รวม)
df_survey['Image 1'] = SURVEY_IMG_DIR + df_survey['Image 1']
df_survey['Image 2'] = SURVEY_IMG_DIR + df_survey['Image 2']

# สร้าง Full Path ให้รูปภาพ Instagram (ต้องแทรกชื่อ Menu เข้าไปใน Path)
df_ig['Image 1'] = IG_IMG_DIR + df_ig['Menu'] + '/' + df_ig['Image 1']
df_ig['Image 2'] = IG_IMG_DIR + df_ig['Menu'] + '/' + df_ig['Image 2']

# รวม Data เข้าด้วยกัน
df_all = pd.concat([df_survey[['Image 1', 'Image 2', 'Menu', 'Winner']], 
                    df_ig[['Image 1', 'Image 2', 'Menu', 'Winner']]], ignore_index=True)

# แปลง Label (Winner 1 -> 0, Winner 2 -> 1)
df_all['label'] = df_all['Winner'].apply(lambda x: 0 if x == 1 else 1)

# แบ่ง Train/Val (ป้องกัน Leakage และรักษาสัดส่วน Menu)
train_df, val_df = train_test_split(df_all, test_size=0.2, random_state=42, stratify=df_all['Menu'])
print(f"✅ ข้อมูล Train: {len(train_df)} คู่ | Validation: {len(val_df)} คู่")

# ==========================================
# 2. สร้าง tf.data.Dataset Pipeline
# ==========================================
def load_and_preprocess_image(path1, path2, label):
    def process_img(path):
        img = tf.io.read_file(path)
        img = tf.image.decode_jpeg(img, channels=3)
        img = tf.image.resize(img, [224, 224])
        return img
    
    img1 = process_img(path1)
    img2 = process_img(path2)
    return ({"image_1": img1, "image_2": img2}, label)

def create_dataset(dataframe, batch_size=32, is_training=False):
    dataset = tf.data.Dataset.from_tensor_slices((
        dataframe['Image 1'].values,
        dataframe['Image 2'].values,
        dataframe['label'].values
    ))
    dataset = dataset.map(load_and_preprocess_image, num_parallel_calls=tf.data.AUTOTUNE)
    if is_training:
        dataset = dataset.shuffle(buffer_size=1000)
    return dataset.batch(batch_size).prefetch(tf.data.AUTOTUNE)

BATCH_SIZE = 32
train_dataset = create_dataset(train_df, batch_size=BATCH_SIZE, is_training=True)
val_dataset = create_dataset(val_df, batch_size=BATCH_SIZE, is_training=False)

# ==========================================
# 3. สร้างสถาปัตยกรรม Siamese Network
# ==========================================
def build_siamese_model(input_shape=(224, 224, 3)):
    base_model = EfficientNetB0(weights='imagenet', include_top=False, input_shape=input_shape)
    base_model.trainable = False # Freeze weights ไว้ก่อน
    
    inputs = Input(shape=input_shape)
    x = base_model(inputs, training=False)
    x = GlobalAveragePooling2D()(x)
    feature_extractor = Model(inputs, x, name="feature_extractor")

    input_1 = Input(shape=input_shape, name="image_1")
    input_2 = Input(shape=input_shape, name="image_2")
    
    feat_1 = feature_extractor(input_1)
    feat_2 = feature_extractor(input_2)
    
    merged = Concatenate()([feat_1, feat_2])
    x = Dense(256, activation='relu')(merged)
    x = Dropout(0.3)(x)
    x = Dense(128, activation='relu')(x)
    outputs = Dense(1, activation='sigmoid', name="output")(x)
    
    model = Model(inputs=[input_1, input_2], outputs=outputs, name="siamese_network")
    return model

model = build_siamese_model()
model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.001), 
              loss='binary_crossentropy', 
              metrics=['accuracy'])

# ==========================================
# 4. สั่งเทรนโมเดล (Training Loop)
# ==========================================
# ตั้งค่า Callbacks ให้เซฟโมเดลที่ดีที่สุดและหยุดเทรนถ้าเริ่ม Overfit
callbacks = [
    ModelCheckpoint('best_siamese_model.h5', save_best_only=True, monitor='val_accuracy', mode='max', verbose=1),
    EarlyStopping(monitor='val_accuracy', patience=5, restore_best_weights=True, verbose=1)
]

print("🚀 เริ่มเทรนโมเดล...")
history = model.fit(
    train_dataset,
    validation_data=val_dataset,
    epochs=20, # ปรับเพิ่มลดได้
    callbacks=callbacks
)
print("🎉 เทรนเสร็จสิ้น! โมเดลที่แม่นที่สุดถูกเซฟไว้ในชื่อ 'best_siamese_model.h5'")

: 